# YOLO Foundations Part 3: Custom Training (Tutorial) 🎯

In this lab, you will learn how to fine-tune a YOLO model on a custom dataset. Fine-tuning lets you take a model that already "understands" the world and specialize it for your own task.

For this tutorial we use **COCO8** — a tiny 8-image slice of COCO that ships with Ultralytics. It is *not* meant to produce a great model; it is meant to make every cell run end-to-end in under two minutes so you can focus on the **process**, not the metrics.

### Learning Objectives
- Understand the structure of a YOLO dataset (`.yaml`).
- Fine-tune a pretrained YOLO26 model on a custom dataset.
- Learn the concept of **Transfer Learning** and **Layer Freezing**.
- Analyze training logs and mAP metrics.
- **Export** your trained model for deployment.

## 1. Environment Setup 🛠️

First, we ensure we have the `ultralytics` package installed and check our hardware (CPU/GPU).

In [ ]:
!pip install  -q ultralytics

In [ ]:
import ultralytics
from ultralytics import YOLO
import matplotlib.pyplot as plt
import cv2
import os

ultralytics.checks()

## 2. The Dataset: COCO8 📁

YOLO datasets are defined by a `.yaml` file. It contains:
- Paths to `train` and `val` image directories.
- The number of classes (`nc`).
- The names of the classes (`names`).

**COCO8** is a tiny demo dataset bundled with Ultralytics — 8 training images and 8 validation images sampled from COCO, covering 80 classes (person, car, dog, etc.). It auto-downloads the first time you train on it.

> 🔎 **Explore first.** Before training, skim the dataset card to see the classes and sample images:
> 👉 https://docs.ultralytics.com/datasets/detect/coco8/

In [ ]:
data_path = "coco8.yaml"
print(f"Using dataset: {data_path}")

## 3. Training the Model 🚀

We will start with a pretrained `yolo26n.pt` (Nano) model and train it for 3 epochs.

**Key Arguments:**
- `data`: Path to the `.yaml` file.
- `epochs`: Number of times to see the full dataset.
- `imgsz`: Image size (multiples of 32).
- `project` & `name`: Where to save the results.

In [ ]:
# Initialize a YOLO model (starting from pretrained weights)
model = YOLO("yolo26n.pt")

# Start training
results = model.train(
    data="coco8.yaml",
    epochs=3,
    imgsz=640,
    project="coco8_training",
    name="demo_run"
)

## 4. Advanced: Transfer Learning & Freezing ❄️

In Transfer Learning, we often **freeze** the early layers of the network (the "backbone") because they already know how to detect basic shapes and edges. This speeds up training and prevents the model from "forgetting" general features.

In [ ]:
# Example: Freeze the first 10 layers (backbone)
model = YOLO("yolo26n.pt")
model.train(
    data="coco8.yaml",
    epochs=2,
    freeze=10,
    project="coco8_training",
    name="frozen_backbone"
)

## 5. Visualizing Results 📈

After training, YOLO saves plots of our metrics. Let's look at the training loss and mAP curves.

In [ ]:
from IPython.display import Image, display

# Path to the results image saved by YOLO
results_img = "runs/detect/coco8_training/demo_run/results.png"

if os.path.exists(results_img):
    display(Image(filename=results_img, width=800))
else:
    print("Results image not found. Did the training finish?")

## 6. Inference Challenge 🔍

Now, use the trained model to detect objects in a new image.

In [ ]:
# Load best weights
best_model = YOLO("runs/detect/coco8_training/demo_run/weights/best.pt")

# Predict
results = best_model.predict("https://ultralytics.com/images/bus.jpg", conf=0.25)

# Plot
res_plotted = results[0].plot()
plt.imshow(res_plotted[:, :, ::-1])
plt.axis('off')
plt.show()

## 7. Exporting your Model 📦

Once you are happy with your model, you usually want to export it to a format optimized for deployment, such as **ONNX**.

In [ ]:
best_model.export(format='onnx')

---
## Summary ✅

Congratulations! You've walked through the full custom-training loop on the COCO8 demo dataset. You've learned how to:
1.  **Prepare** a training environment.
2.  **Fine-tune** a YOLO26 model on a YAML-defined dataset.
3.  Use **Layer Freezing** for faster training.
4.  **Analyze** performance and **Export** the model.

➡️ Next, head to the exercise notebook to apply this on a real wildlife dataset.